In [2]:
# Import required libraries
import warnings
from datetime import datetime, timedelta

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import clear_output, display

warnings.filterwarnings("ignore")

import sys

sys.path.append("../backtest")
# Import strategies
from strategies import STRATEGIES

In [3]:
stock = yf.Ticker("SPY")
data = stock.history(start="2022-01-01", end="2022-12-31")
stock.option_chain().calls

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency
0,SPY260105C00510000,2026-01-02 17:30:06+00:00,510.0,172.37,172.07,174.86,0.020004,0.011607,2.0,0,1.733400,True,REGULAR,USD
1,SPY260105C00540000,2025-12-31 19:32:06+00:00,540.0,145.00,142.07,144.86,0.000000,0.000000,NaN,1,1.429690,True,REGULAR,USD
2,SPY260105C00550000,2025-12-24 16:49:02+00:00,550.0,141.23,132.08,134.85,0.000000,0.000000,NaN,1,1.331058,True,REGULAR,USD
3,SPY260105C00595000,2026-01-02 20:44:46+00:00,595.0,87.82,87.09,89.86,-2.510002,-2.778703,1.0,1,0.904298,True,REGULAR,USD
4,SPY260105C00605000,2026-01-02 15:12:00+00:00,605.0,80.07,77.08,79.88,-1.050003,-1.294382,3.0,0,0.812502,True,REGULAR,USD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,SPY260105C00775000,2025-12-26 21:07:01+00:00,775.0,0.01,0.00,0.01,0.000000,0.000000,25.0,26,0.531255,False,REGULAR,USD
85,SPY260105C00785000,2025-12-23 20:54:10+00:00,785.0,0.01,0.00,0.01,0.000000,0.000000,NaN,49,0.578129,False,REGULAR,USD
86,SPY260105C00790000,2025-12-29 17:32:39+00:00,790.0,0.02,0.00,0.01,0.000000,0.000000,1.0,19,0.593754,False,REGULAR,USD
87,SPY260105C00795000,2025-12-26 20:02:42+00:00,795.0,0.01,0.00,0.01,0.000000,0.000000,21.0,79,0.625004,False,REGULAR,USD


In [4]:
def fetch_stock_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch stock data from Yahoo Finance.
    
    Args:
        ticker: Stock symbol (e.g., 'AAPL')
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
    
    Returns:
        DataFrame with stock data
    """
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(start=start_date, end=end_date)
        
        if data.empty:
            raise ValueError(f"No data found for {ticker}")
        
        # Reset index to make Date a column
        data = data.reset_index()
        
        return data
    
    except Exception as e:
        raise Exception(f"Error fetching data: {str(e)}")

In [5]:
def plot_results(data: pd.DataFrame, results: dict, ticker: str, strategy_name: str, 
                 start_date: str, end_date: str):
    """
    Plot stock price and trading performance.
    
    Args:
        data: Stock data DataFrame
        results: Strategy execution results
        ticker: Stock ticker symbol
        strategy_name: Name of strategy used
        start_date: Requested start date
        end_date: Requested end date
    """
    fig, ax = plt.subplots(1, 1, figsize=(16, 9))
    
    # Plot stock price with gradient fill
    ax.plot(data['Date'], data['Close'], linewidth=2.5, label='Close Price', 
            color='#2E86AB', zorder=3)
    ax.fill_between(data['Date'], data['Low'], data['High'], alpha=0.15, 
                    color='#A23B72', label='Daily Range')
    
    # Mark buy transactions
    buy_txns = [t for t in results['transactions'] if t['type'] == 'BUY']
    if buy_txns:
        buy_dates = [t['date'] for t in buy_txns]
        buy_prices = [t['price'] for t in buy_txns]
        ax.scatter(buy_dates, buy_prices, color='#00C853', s=120, marker='^', 
                   label='Buy', zorder=5, alpha=0.8, edgecolors='darkgreen', linewidth=1.5)
    
    # Mark sell transactions
    sell_txns = [t for t in results['transactions'] if t['type'] == 'SELL']
    if sell_txns:
        sell_dates = [t['date'] for t in sell_txns]
        sell_prices = [t['price'] for t in sell_txns]
        ax.scatter(sell_dates, sell_prices, color='#FF1744', s=120, marker='v', 
                   label='Sell', zorder=5, alpha=0.8, edgecolors='darkred', linewidth=1.5)
    
    # Title and labels
    ax.set_title(f'{ticker} - {strategy_name} Strategy Performance', 
                fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax.set_ylabel('Price ($)', fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.7)
    
    # Add performance metrics as text box
    gain_color = '#00C853' if results['gain_amount'] >= 0 else '#FF1744'
    metrics_text = (
        f"Initial Portfolio Value: ${results['total_cost']:,.2f}\n"
        f"Final Portfolio Value: ${results['final_value']:,.2f}\n"
        f"Gain/Loss: ${results['gain_amount']:,.2f} ({results['gain_pct']:+.2f}%)"
    )
    
    # Position text box in top right
    props = dict(boxstyle='round,pad=0.8', facecolor='white', 
                 edgecolor=gain_color, linewidth=2, alpha=0.95)
    ax.text(0.98, 0.97, metrics_text, transform=ax.transAxes, 
            fontsize=11, verticalalignment='top', horizontalalignment='right',
            bbox=props, fontweight='bold', family='monospace')
    
    # Improve aesthetics
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*70)
    print(f"STRATEGY: {strategy_name}")
    print("="*70)
    print(f"Ticker:           {ticker}")
    print(f"Period:           {start_date} to {end_date}")
    print(f"Actual Data:      {data['Date'].iloc[0].date()} to {data['Date'].iloc[-1].date()}")
    print(f"Total Investment: ${results['total_cost']:,.2f}")
    print(f"Final Value:      ${results['final_value']:,.2f}")
    print(f"Gain/Loss:        ${results['gain_amount']:,.2f} ({results['gain_pct']:+.2f}%)")
    print(f"Shares Owned:     {results['shares']:.4f}")
    print(f"Transactions:     {len(results['transactions'])}")
    print("="*70)

In [6]:
def analyze_stock(ticker, start_date, end_date, amount, strategy_name):
    """
    Main analysis function.
    
    Args:
        ticker: Stock ticker symbol
        start_date: Start date string
        end_date: End date string  
        amount: Investment amount
        strategy_name: Strategy to use
    """
    output.clear_output()
    
    with output:
        try:
            print(f"Fetching data for {ticker}...")
            data = fetch_stock_data(ticker, start_date, end_date)
            
            print(f"Executing {strategy_name} strategy...")
            strategy = STRATEGIES[strategy_name]
            results = strategy.execute(data, amount)
            
            print("\nGenerating visualizations...\n")
            plot_results(data, results, ticker, strategy_name, start_date, end_date)
            
        except Exception as e:
            print(f"❌ Error: {str(e)}")
            print("\nPlease check:")
            print("- Ticker symbol is valid")
            print("- Date range has trading days")
            print("- Amount is positive")

## Interactive Analysis Interface

Configure your analysis parameters below and click **Analyze** to run.

In [7]:
# Create UI widgets
ticker_input = widgets.Text(
    value='AAPL',
    description='Ticker:',
    placeholder='e.g., AAPL, MSFT, GOOGL',
    style={'description_width': '120px'}
)

start_date_input = widgets.DatePicker(
    description='Start Date:',
    value=datetime.now() - timedelta(days=365),
    style={'description_width': '120px'}
)

end_date_input = widgets.DatePicker(
    description='End Date:',
    value=datetime.now(),
    style={'description_width': '120px'}
)

amount_input = widgets.FloatText(
    value=10000,
    description='Amount ($):',
    min=0,
    step=100,
    style={'description_width': '120px'}
)

strategy_dropdown = widgets.Dropdown(
    options=list(STRATEGIES.keys()),
    description='Strategy:',
    style={'description_width': '120px'}
)

analyze_button = widgets.Button(
    description='Analyze',
    button_style='success',
    icon='chart-line',
    layout=widgets.Layout(width='200px', height='40px')
)

output = widgets.Output()

# Button click handler
def on_analyze_clicked(b):
    analyze_stock(
        ticker=ticker_input.value.upper().strip(),
        start_date=start_date_input.value.strftime('%Y-%m-%d'),
        end_date=end_date_input.value.strftime('%Y-%m-%d'),
        amount=amount_input.value,
        strategy_name=strategy_dropdown.value
    )

analyze_button.on_click(on_analyze_clicked)

# Display UI
ui = widgets.VBox([
    widgets.HTML("<h3>📊 Stock Analysis Configuration</h3>"),
    ticker_input,
    start_date_input,
    end_date_input,
    amount_input,
    strategy_dropdown,
    analyze_button,
    output
], layout=widgets.Layout(padding='20px', border='2px solid #ddd', border_radius='10px'))

display(ui)

## Strategy Descriptions

**DCA Every Friday**: Divides total investment equally across all Fridays in the date range. Reduces timing risk through regular purchases.

**Buy All At Once**: Invests the full amount on the first trading day. Simple lump sum approach.

**Intraday (4PM-9AM)**: Buys at market close (4PM) and sells at next day's open (9:30AM). Compounds gains/losses daily. Uses available data as proxy for intraday prices.

**Earnings Play**: Buys 14 days before earnings and sells 1 day before earnings announcement. Divides investment equally across all earnings cycles in the date range. Simulates quarterly earnings (~every 90 days).